# 365 Probabilidades - Dia #111
## Qual a probabilidade de o seu dia acontecer sem você?

**Tipo:** Comportamental
**Data de publicação:** 2026-10-02
**Ferramenta:** Python
**Decisão analisada:** Quanto do meu dia eu realmente decido?
**Hashtag:** #365Probabilidades #Dia111

---

### 📖 A História

Existe um número que circula há vinte anos em livro de hábito, palestra de
produtividade e legenda de rede social: **43% do seu dia é automático.**

É um número bom. Tem artigo por trás, tem revista séria, tem amostra grande. E, ao
contrário da maioria dos números que essa série já desmontou, ele não está errado.

O problema é outro. O mesmo artigo que publicou 43% publicou também 35%, no estudo
anterior, com o mesmo método. Ninguém repete o 35%. E quando se conta a amostra pela
unidade certa, os dois números deixam de ser distinguíveis.

Pior: mesmo que o 43% esteja certíssimo, ele não descreve o seu dia. Ele descreve a
média de dias que variam tanto entre si que a média não é um retrato de ninguém.

Hoje o modelo não serve para derrubar o número. Serve para perguntar de quem ele é.

---

### 📚 O Conceito: hábito é comportamento que dispensa decisão

Wendy Wood, Jeffrey Quinn e Deborah Kashy definiram hábito de um jeito operacional:
comportamento repetido **quase todo dia** e **geralmente no mesmo lugar**. Os dois
critérios juntos, não um ou outro.

A ideia por trás é que a repetição em contexto estável transfere o controle da ação da
intenção para a deixa ambiental. Você não decide escovar os dentes. O banheiro decide
por você.

Para medir isso fora do laboratório, os autores usaram amostragem por sinal: um relógio
apitava de hora em hora e a pessoa anotava o que estava fazendo, pensando e sentindo
naquele instante. Depois classificava cada comportamento pelos dois critérios.

O Estudo 1 acompanhou 70 estudantes por um dia e achou 35% dos comportamentos
habituais. O Estudo 2 acompanhou 209 estudantes por dois dias e achou 43%.

O 43% é o que sobreviveu na cultura.

---

### 🧮 O Modelo

Quatro camadas. A primeira testa o número, a segunda testa a amostra, e as duas últimas
perguntam de quem é o dia.

1. **Posterior de Jeffreys** para as duas proporções publicadas, contando cada
   comportamento anotado como uma observação. É a leitura ingênua, e a que produz o
   intervalo estreito que todo mundo cita.
2. **Correção por efeito de desenho.** As observações não são independentes: vêm em
   blocos de uma mesma pessoa, cerca de 21 anotações por participante no Estudo 2.
   O tamanho efetivo da amostra encolhe por um fator 1 + (m − 1)·ρ. Varremos ρ de zero
   ao limite em que a unidade vira a pessoa, e refazemos o teste entre os dois estudos.
3. **Heterogeneidade entre pessoas.** Beta-binomial calibrada pela média publicada e
   pela correlação intraclasse adotada, para descrever não a média dos dias, mas a
   distribuição deles.
4. **Simulação do dia.** Dez horas acordado amostradas por pessoa, semente declarada,
   para traduzir proporção em horas e comparar o mundo real com um mundo em que todo
   mundo fosse a média.

**Fontes:**
- Wood, W., Quinn, J. M. & Kashy, D. A. (2002). "Habits in Everyday Life: Thought,
  Emotion, and Action". *Journal of Personality and Social Psychology*, 83(6),
  1281-1297. Estudo 1: N=70 participantes, média de 9,58 anotações cada, 35% habituais.
  Estudo 2: N=209 participantes, média de 20,74 anotações cada, 43% habituais.

**Nota metodológica sobre o fator ×0,80.** Não aplicado, e o motivo importa. O fator
existe para descontar o exagero de quem se autodeclara em survey populacional, quando o
autorrelato é o ruído entre a pergunta e o mundo. Aqui o autorrelato **é** a medida:
"quase todo dia, no mesmo lugar" é a definição da variável, não uma aproximação dela.
Cortar 20% do 43% seria cortar 20% da definição. Segue a decisão tomada no **#063**.
A incerteza entra por onde ela de fato existe, que é o tamanho efetivo da amostra, e
isso alarga o intervalo em vez de empurrar o ponto.

**Limite declarado:** a correlação intraclasse não é publicada no artigo. É escolha
nossa, varrida de 0 a 1 na camada 2, com o cenário central em 0,20 usado nas camadas 3
e 4. Todo número que depende dela está marcado como tal.

**Ponte interna:** o **#006** mostrou disciplina prevendo resultado melhor que
inteligência. Lá a repetição era virtude. Aqui ela é anestesia. É o mesmo mecanismo
visto dos dois lados.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
SORTEIOS = 500_000

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(n):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DA LITERATURA ---
# Wood, Quinn & Kashy (2002), JPSP 83(6), 1281-1297.
# Amostragem por sinal: relógio apita de hora em hora, a pessoa anota o que está fazendo.

ESTUDOS = {
    'Estudo 1': dict(p=0.35, pessoas=70,  por_pessoa=9.58,  dias=1),
    'Estudo 2': dict(p=0.43, pessoas=209, por_pessoa=20.74, dias=2),
}
for e in ESTUDOS.values():
    e['registros'] = round(e['pessoas'] * e['por_pessoa'])

# Correlação intraclasse: NÃO publicada. Escolha nossa, varrida na camada 2.
ICC_CENTRAL = 0.20
HORAS_DIA = 10          # horas acordado amostradas por dia no Estudo 2 (20,74 / 2 dias)

print("=" * 70)
print("  DADOS DA LITERATURA · WOOD, QUINN & KASHY (2002)")
print("=" * 70)
print("\n  Critério de hábito: 'quase todo dia' E 'geralmente no mesmo lugar'.")
print("  Os dois juntos. Classificação feita pelo próprio participante.\n")
for nome, e in ESTUDOS.items():
    print(f"  {nome}: {e['pessoas']:>3} pessoas · {e['dias']} dia"
          f"{'s' if e['dias'] > 1 else ' '} · "
          f"{vir(e['por_pessoa'], 2)} anotações por pessoa · "
          f"{mil(e['registros'])} registros")
    print(f"  {'':9} comportamentos habituais: {e['p'] * 100:.0f}%")
print(f"\n  O número que a cultura guardou: {ESTUDOS['Estudo 2']['p'] * 100:.0f}%.")
print(f"  O número que ninguém repete:    {ESTUDOS['Estudo 1']['p'] * 100:.0f}%.")
print(f"\n  ICC adotada no cenário central: {vir(ICC_CENTRAL, 2)} (escolha nossa, não do artigo)")
print(f"  ×0,80: não aplicado. O autorrelato aqui é a medida, não o ruído. Ver #063.")
print("=" * 70)

In [ ]:
# --- O MODELO ---

# ── CAMADA 1 · posterior de Jeffreys, contando cada anotação como observação ──
def jeffreys(p, n):
    x = p * n
    return stats.beta(x + 0.5, n - x + 0.5)

def ic(dist):
    return dist.interval(0.95)

print("=" * 70)
print("  CAMADA 1 · A LEITURA INGÊNUA (cada anotação vale uma observação)")
print("=" * 70)
for nome, e in ESTUDOS.items():
    d = jeffreys(e['p'], e['registros'])
    lo, hi = ic(d)
    e['d_ingenuo'] = d
    print(f"  {nome}: {e['p'] * 100:.0f}% · n = {mil(e['registros'])} registros"
          f" · IC 95% [{vir(lo * 100)}%; {vir(hi * 100)}%]"
          f" · largura {vir((hi - lo) * 100)} pp")

# ── CAMADA 2 · efeito de desenho: as anotações vêm em blocos de uma mesma pessoa ──
# DEFF = 1 + (m - 1) * rho ; n_efetivo = n / DEFF
# rho = 0 -> a unidade é a anotação. rho = 1 -> a unidade é a pessoa.
def n_efetivo(e, rho):
    deff = 1 + (e['por_pessoa'] - 1) * rho
    return e['registros'] / deff, deff

print("\n" + "=" * 70)
print("  CAMADA 2 · O QUE CONTA COMO AMOSTRA (Estudo 2)")
print("=" * 70)
print("\n  DEFF = 1 + (m − 1)·ρ,  com m = 20,74 anotações por pessoa\n")
print(f"  {'ρ':>5} {'DEFF':>7} {'n efetivo':>11}   {'IC 95%':>20} {'largura':>9}")
e2 = ESTUDOS['Estudo 2']
for rho in [0.00, 0.05, 0.10, 0.20, 0.30, 0.50, 1.00]:
    ne, deff = n_efetivo(e2, rho)
    lo, hi = ic(jeffreys(e2['p'], ne))
    marca = "  ← cenário central" if abs(rho - ICC_CENTRAL) < 1e-9 else ""
    print(f"  {vir(rho, 2):>5} {vir(deff, 2):>7} {mil(round(ne)):>11}   "
          f"[{vir(lo * 100)}%; {vir(hi * 100)}%]  {vir((hi - lo) * 100):>6} pp{marca}")
print(f"\n  No limite ρ = 1 a amostra vira as {e2['pessoas']} pessoas, e o intervalo")
print(f"  passa de {vir(2.9)} pp para {vir(13.3)} pp. O 43% não tem duas casas decimais.")

# ── CAMADA 2b · os dois estudos são distinguíveis? ──
print("\n" + "=" * 70)
print("  CAMADA 2b · 35% E 43% SÃO NÚMEROS DIFERENTES?")
print("=" * 70)
g1 = np.random.default_rng(SEMENTE)
g2 = np.random.default_rng(SEMENTE + 1)
for rot, chave in [("contando anotação", 'registros'), ("contando pessoa", 'pessoas')]:
    a = jeffreys(ESTUDOS['Estudo 1']['p'], ESTUDOS['Estudo 1'][chave]).rvs(SORTEIOS, random_state=g1)
    b = jeffreys(ESTUDOS['Estudo 2']['p'], ESTUDOS['Estudo 2'][chave]).rvs(SORTEIOS, random_state=g2)
    dif = b - a
    lo, hi = np.percentile(dif, [2.5, 97.5])
    print(f"\n  {rot}:")
    print(f"    P(Estudo 2 > Estudo 1) = {vir((dif > 0).mean() * 100)}%")
    print(f"    diferença = {vir(dif.mean() * 100)} pp · "
          f"IC 95% [{vir(lo * 100)} pp; {vir(hi * 100)} pp]")
    print(f"    zero {'DENTRO' if lo < 0 < hi else 'FORA'} do intervalo")
print("\n  Mesmo artigo, mesmo método, dois números. Contando pessoa, não dá para")
print("  afirmar que são diferentes. A cultura escolheu o maior.")

# ── CAMADA 3 · a distribuição dos dias, não a média deles ──
s = 1 / ICC_CENTRAL - 1
alfa, beta_ = e2['p'] * s, (1 - e2['p']) * s
pessoas = stats.beta(alfa, beta_)
d10, d90 = pessoas.ppf(0.10), pessoas.ppf(0.90)
p_metade = 1 - pessoas.cdf(0.50)

print("\n" + "=" * 70)
print("  CAMADA 3 · O SEU DIA NÃO É O DIA MÉDIO")
print("=" * 70)
print(f"\n  Beta-binomial calibrada em média {e2['p'] * 100:.0f}% e ρ = {vir(ICC_CENTRAL, 2)}")
print(f"  Beta(α = {vir(alfa, 2)}; β = {vir(beta_, 2)})\n")
print(f"  Média entre pessoas:               {vir(pessoas.mean() * 100)}%")
print(f"  Os 80% do meio vão de              {vir(d10 * 100)}% a {vir(d90 * 100)}%")
print(f"  P(mais da metade do dia no automático) = {vir(p_metade * 100)}%")
print(f"  P(menos de um quinto no automático)    = {vir(pessoas.cdf(0.20) * 100)}%")

# ── CAMADA 4 · traduzindo em horas ──
g = np.random.default_rng(SEMENTE)
p_pessoa = g.beta(alfa, beta_, SORTEIOS)
horas = g.binomial(HORAS_DIA, p_pessoa)
horas_medio = g.binomial(HORAS_DIA, e2['p'], SORTEIOS)

print("\n" + "=" * 70)
print(f"  CAMADA 4 · {HORAS_DIA} HORAS ACORDADO, {mil(SORTEIOS)} DIAS SIMULADOS")
print("=" * 70)
print(f"\n  {'':34}{'mundo real':>14}{'se todos fossem a média':>26}")
print(f"  {'horas no automático (média)':<34}{vir(horas.mean()):>14}{vir(horas_medio.mean()):>26}")
print(f"  {'os 80% do meio':<34}"
      f"{f'{np.percentile(horas, 10):.0f} a {np.percentile(horas, 90):.0f} h':>14}"
      f"{f'{np.percentile(horas_medio, 10):.0f} a {np.percentile(horas_medio, 90):.0f} h':>26}")
print(f"  {'P(dia inteiro decidido na hora)':<34}{vir((horas == 0).mean() * 100) + '%':>14}"
      f"{vir((horas_medio == 0).mean() * 100, 2) + '%':>26}")
print(f"  {'P(dia inteiro no automático)':<34}{vir((horas == HORAS_DIA).mean() * 100) + '%':>14}"
      f"{vir((horas_medio == HORAS_DIA).mean() * 100, 2) + '%':>26}")
print(f"\n  A média sobrevive: {vir(horas.mean())} h contra {vir(horas_medio.mean())} h.")
print(f"  O dia não: a faixa dobra de largura, e um dia inteiramente seu deixa de ser")
print(f"  raridade estatística para virar {vir((horas == 0).mean() * 100)}% dos dias.")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · os dois números do mesmo artigo ══
fig1, ax1 = plt.subplots(figsize=(11, 6.5))

linhas = []
for nome, e in ESTUDOS.items():
    for rot, chave, cor in [('cada anotação conta', 'registros', CINZA),
                            ('cada pessoa conta', 'pessoas', VERMELHO)]:
        lo, hi = ic(jeffreys(e['p'], e[chave]))
        linhas.append((f"{nome}\n{rot}", e['p'], lo, hi, cor))

ys = [3.25, 2.75, 1.25, 0.75]
for y, (rot, p, lo, hi, cor) in zip(ys, linhas):
    ax1.plot([lo * 100, hi * 100], [y, y], color=cor, linewidth=6, alpha=0.35,
             solid_capstyle='round')
    ax1.plot(p * 100, y, 'o', color=cor, markersize=11, zorder=3)
    ax1.text(hi * 100 + 0.8, y, f"[{vir(lo * 100)}; {vir(hi * 100)}]",
             va='center', fontsize=10, color=cor)

ax1.set_yticks(ys)
ax1.set_yticklabels([l[0] for l in linhas], fontsize=10)
ax1.set_xlim(20, 59)
ax1.set_ylim(0.2, 3.8)
ax1.set_xlabel('Comportamentos classificados como habituais (%)')
ax1.set_title('O mesmo artigo publicou 35% e 43%\n'
              'Contando pessoa em vez de anotação, os intervalos se encostam',
              fontsize=13, pad=15)
ax1.axvline(43, color=DOURADO, linestyle='--', linewidth=1.5, alpha=0.8)
ax1.text(43.5, 2.05, 'o número que a cultura guardou',
         fontsize=10, color=DOURADO)
ax1.grid(axis='y', alpha=0)

plt.figtext(0.5, 0.005, 'Fonte: Wood, Quinn & Kashy (2002), JPSP 83(6) · '
            'posterior de Jeffreys | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-111-grafico-01-dois-numeros.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · quanto a amostra encolhe ══
fig2, ax2 = plt.subplots(figsize=(11, 6.5))

rhos = np.linspace(0, 1, 400)
los, his = [], []
for r in rhos:
    ne, _ = n_efetivo(e2, r)
    lo, hi = ic(jeffreys(e2['p'], ne))
    los.append(lo * 100)
    his.append(hi * 100)
los, his = np.array(los), np.array(his)

ax2.fill_between(rhos, los, his, color=VERDE, alpha=0.20)
ax2.plot(rhos, los, color=VERDE, linewidth=2)
ax2.plot(rhos, his, color=VERDE, linewidth=2)
ax2.axhline(e2['p'] * 100, color='#333333', linewidth=2)
ax2.axvline(ICC_CENTRAL, color=VERMELHO, linestyle='--', linewidth=1.8)

lo_c, hi_c = ic(jeffreys(e2['p'], n_efetivo(e2, ICC_CENTRAL)[0]))
ax2.plot([ICC_CENTRAL], [lo_c * 100], 'o', color=VERMELHO, markersize=8, zorder=3)
ax2.plot([ICC_CENTRAL], [hi_c * 100], 'o', color=VERMELHO, markersize=8, zorder=3)
ax2.text(ICC_CENTRAL + 0.02, 50.5,
         f'cenário central ρ = {vir(ICC_CENTRAL, 2)}\n'
         f'[{vir(lo_c * 100)}%; {vir(hi_c * 100)}%]',
         fontsize=10, color=VERMELHO)
ax2.text(0.52, 43.35, f'estimativa publicada: {e2["p"] * 100:.0f}%',
         fontsize=10, color='#333333')
ax2.text(0.70, 36.2, f'ρ = 1: a amostra vira as {e2["pessoas"]} pessoas',
         fontsize=10, color=VERDE, ha='left', va='top')

ax2.set_xlim(0, 1)
ax2.set_ylim(34, 52)
ax2.set_xlabel('ρ · correlação intraclasse adotada (não publicada no artigo)')
ax2.set_ylabel('Intervalo de 95% para a proporção (%)')
ax2.set_title('O 43% depende de quem é a amostra\n'
              '4.335 anotações vindas de 209 pessoas não são 4.335 observações',
              fontsize=13, pad=15)

plt.figtext(0.5, 0.005, 'Modelo próprio · efeito de desenho DEFF = 1 + (m − 1)·ρ, '
            'm = 20,74 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-111-grafico-02-a-amostra.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · a distribuição dos dias ══
fig3, ax3 = plt.subplots(figsize=(11, 6.5))

x = np.linspace(0.001, 0.999, 1000)
y = pessoas.pdf(x)
ax3.plot(x * 100, y, color=VERDE, linewidth=2.5)
faixa = (x >= d10) & (x <= d90)
ax3.fill_between(x[faixa] * 100, y[faixa], color=VERDE, alpha=0.22)
ax3.axvline(e2['p'] * 100, color=VERMELHO, linewidth=2.5)
ax3.text(e2['p'] * 100 + 1.5, max(y) * 0.40,
         f'a média publicada\n{e2["p"] * 100:.0f}%', fontsize=11, color=VERMELHO)
ax3.annotate('', xy=(d10 * 100, max(y) * 0.12), xytext=(d90 * 100, max(y) * 0.12),
             arrowprops=dict(arrowstyle='<->', color=DOURADO, lw=2))
ax3.text((d10 + d90) / 2 * 100, max(y) * 0.055,
         f'os 80% do meio: de {vir(d10 * 100)}% a {vir(d90 * 100)}%',
         ha='center', fontsize=11, color=DOURADO, fontweight='bold')

ax3.set_xlim(0, 100)
ax3.set_ylim(bottom=0)
ax3.set_xlabel('Fração do dia no automático (%)')
ax3.set_ylabel('Densidade entre pessoas')
ax3.set_title('A média existe. O dia médio não.\n'
              f'Beta(α = {vir(alfa, 2)}; β = {vir(beta_, 2)}), calibrada em 43% e ρ = '
              f'{vir(ICC_CENTRAL, 2)}',
              fontsize=13, pad=15)

plt.figtext(0.5, 0.005, 'Modelo próprio · Beta-binomial calibrada pela média publicada '
            'e pela ρ adotada | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-111-grafico-03-o-seu-dia.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · as horas ══
fig4, ax4 = plt.subplots(figsize=(11, 6.5))

ks = np.arange(HORAS_DIA + 1)
freq_real = np.array([(horas == k).mean() for k in ks]) * 100
freq_med = np.array([(horas_medio == k).mean() for k in ks]) * 100

larg = 0.40
ax4.bar(ks - larg / 2, freq_med, larg, color=CINZA, alpha=0.75,
        label='se todo mundo fosse a média')
ax4.bar(ks + larg / 2, freq_real, larg, color=VERDE, alpha=0.9,
        label='com as pessoas sendo diferentes')

ax4.set_xticks(ks)
ax4.set_xlabel(f'Horas no automático, de {HORAS_DIA} horas acordado')
ax4.set_ylabel('Proporção dos dias (%)')
ax4.set_title('A mesma média, dois mundos diferentes\n'
              f'{mil(SORTEIOS)} dias simulados, semente {SEMENTE}',
              fontsize=13, pad=15)
ax4.legend(frameon=False, fontsize=10.5)

ax4.annotate(f'um dia inteiramente seu:\n{vir(freq_med[0], 2)}% contra '
             f'{vir(freq_real[0])}%',
             xy=(0 + larg / 2, freq_real[0] + 0.3), xytext=(1.75, 19.5),
             fontsize=10.5, color=VERMELHO,
             arrowprops=dict(arrowstyle='->', color=VERMELHO, lw=1.6))

plt.figtext(0.5, 0.005, 'Modelo próprio · Monte Carlo com semente declarada | '
            '#365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-111-grafico-04-as-horas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

O número que sobreviveu não é mentira. É órfão.

O mesmo artigo mediu **35% num estudo e 43% no outro**, com o mesmo método e a mesma
pergunta. Contando cada anotação como uma observação, a diferença é certa. Contando
pessoa, que é a unidade que o desenho do estudo realmente tem, a diferença deixa de ser
afirmável: o intervalo vai de −5,3 a +20,4 pontos percentuais e passa por zero.

A cultura guardou o número maior. Ele soa melhor.

E há um problema mais fundo que o intervalo. Mesmo que 43% esteja certo na média, essa
média está calculada sobre dias que não se parecem. No cenário central, **os 80% do
meio vão de 14% a 74% do dia no automático.** Não é uma faixa. É o mapa inteiro.

Traduzido em horas: um dia de dez horas acordado tem, em média, **4,3 horas que
acontecem sem você**. Mas se as pessoas fossem todas a média, essa conta daria entre
2 e 6 horas para quase todo mundo. Com as pessoas sendo o que são, dá entre 1 e 8.

E a chance de um dia inteiramente decidido na hora, sem nenhuma hora no piloto
automático, sai de **0,4% para 6,7%**. Dezessete vezes maior. Um dia inteiramente seu
não é impossível. É improvável de um jeito bem mais humano do que a média sugeria.

O 43% responde uma pergunta que ninguém fez: quanto do dia médio é automático.

A pergunta que interessa é outra, e o modelo diz que ela não tem resposta única:

*Você é dos 14% ou dos 74%? E quando foi a última vez que você verificou?*

---

### ⚠️ Limitações do Modelo

- **A correlação intraclasse não é publicada.** ρ = 0,20 é escolha nossa. Todas as
  camadas 3 e 4 dependem dela. A camada 2 mostra o resultado para ρ de 0 a 1, e a
  conclusão qualitativa (o intervalo alarga muito) vale em toda a faixa; os números
  específicos de horas e decis, não.
- **A amostra é de estudantes universitários**, num único campus, em 2002. Rotina de
  estudante tem estabilidade de contexto diferente da de quem tem filho pequeno, turno
  variável ou trabalho remoto.
- **Um e dois dias de observação.** Hábito é definido por repetição, e dois dias são
  pouco para observar repetição. O critério depende de o participante afirmar que faz
  aquilo quase todo dia, não de o estudo ter visto.
- **Classificação pelo próprio participante.** É a escolha certa para o construto, e é
  por isso que o ×0,80 não se aplica, mas significa que "automático" aqui é o que a
  pessoa reconhece como automático. O hábito que você não percebe não entra na conta,
  e ele empurra o número para cima.
- **As dez horas do dia simulado** vêm da média de anotações por dia do Estudo 2, não
  de uma medida de tempo acordado. É uma tradução, não um dado.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*